In [0]:
%pip install langchain-huggingface langchain-community langchain-openai faiss-cpu
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:

# Imports
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
import os

/local_disk0/.ephemeral_nfs/envs/pythonEnv-d8d17501-5546-4201-b22d-51aeb4a70639/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


In [0]:

os.enivron["OPEN_AI_KEY"] = " "

In [0]:
# Load Data
df = spark.table("workspace.messy_ecommerce_db.gold_category_revenue")
pdf = df.toPandas()

documents = [
    f"Category {row['category']} generated revenue {row['revenue']}"
    for _, row in pdf.iterrows()
]

In [0]:
# Load Data
df = spark.table("workspace.messy_ecommerce_db.gold_category_revenue")
pdf = df.toPandas()

documents = [
    f"Category {row['category']} generated revenue {row['revenue']}"
    for _, row in pdf.iterrows()
]

In [0]:
# LLM
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0
)

Accessing `__path__` from `.models.aria.image_processing_aria`. Returning `__path__` instead. Behavior may be different and this alias will be removed in future versions.


---------------------------------------------------------------------------
OpenAIError                               Traceback (most recent call last)
File <command-8245222392669673>, line 2
      1 # LLM
----> 2 llm = ChatOpenAI(
      3     model="gpt-3.5-turbo",
      4     temperature=0
      5 )

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-d8d17501-5546-4201-b22d-51aeb4a70639/lib/python3.12/site-packages/langchain_core/load/serializable.py:118, in Serializable.__init__(self, *args, **kwargs)
    116 def __init__(self, *args: Any, **kwargs: Any) -> None:
    117     """"""  # noqa: D419  # Intentional blank docstring
--> 118     super().__init__(*args, **kwargs)

    [... skipping hidden 1 frame]

File /databricks/python/lib/python3.12/site-packages/langchain_openai/chat_models/base.py:990, in BaseChatOpenAI.validate_environment(self)
    980         self.http_async_client = httpx.AsyncClient(
    981             proxy=self.openai_proxy, verify=global_ssl_context
    982      

In [0]:
# RAG Function
def rag_query(query):
    docs = retriever.get_relevant_documents(query)
    
    context = "\n".join([doc.page_content for doc in docs])
    
    prompt = f"""
    You are a data analyst.

    Use the following data to answer the question:

    {context}

    Question: {query}

    Give a clear business answer:
    """
    
    response = llm.invoke(prompt)
    return response.content